In [ ]:
import torch
import torch.nn as nn
import numpy as np
from datasets import load_dataset
from torchvision import transforms
from torch.utils.data import DataLoader, IterableDataset
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class StreamingImageNet(IterableDataset):
    def __init__(self, hf_dataset, transform):
        self.ds = hf_dataset
        self.transform = transform
    def __iter__(self):
        for s in self.ds:
            yield self.transform(s["image"].convert("RGB")), s["label"]

DS = load_dataset("imagenet-1k", split="validation", streaming=True)
dataset = StreamingImageNet(DS, transform)
loader = DataLoader(dataset, batch_size=64, num_workers=4, pin_memory=True)

In [ ]:
initial = nn.Sequential(
    nn.Conv2d(in_channels=3, out_channels=96, stride=2, kernel_size=(7,7), padding=3),
    nn.BatchNorm2d(num_features=96),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=(3,3), stride=2, padding=1)
)
k = 48

class DenseBlock(nn.Module):
    def __init__(self, k, layers, in_layer):
        super().__init__()
        self.layers = nn.ModuleList([])
        for i in range(layers):
            self.layers.append(nn.Sequential(
                nn.BatchNorm2d(num_features=in_layer + i*k),
                nn.ReLU(),
                nn.Conv2d(in_layer + i*k, 4*k, kernel_size=(1,1)),
                nn.BatchNorm2d(4*k),
                nn.ReLU(),
                nn.Conv2d(4*k, k, kernel_size=(3,3), padding=1)
            ))
    def forward(self, x: torch.Tensor):
        for layer in self.layers:
            out = layer(x)
            x = torch.concat([x, out], dim=1)
        return x

class Transition(nn.Module):
    def __init__(self, i):
        super().__init__()
        self.l1 = nn.Conv2d(i, i//2, kernel_size=(1,1))
        self.l2 = nn.AvgPool2d(stride=2, kernel_size=(2,2))
    def forward(self, x):
        return self.l2(self.l1(x))

c0 = 96
c1 = c0 + 6*k
c2 = c1//2 + 12*k
c3 = c2//2 + 36*k
c4 = c3//2 + 24*k

DenseNet161 = nn.Sequential(
    initial,
    DenseBlock(k, 6,  c0),
    Transition(c1),
    DenseBlock(k, 12, c1//2),
    Transition(c2),
    DenseBlock(k, 36, c2//2),
    Transition(c3),
    DenseBlock(k, 24, c3//2),
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Linear(c4, 1000)
).to(device)

In [ ]:
CHECKPOINT_PATH = "/content/drive/MyDrive/densenet161_checkpoint.pt"

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
DenseNet161.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded checkpoint from epoch {checkpoint['epoch']}, batch {checkpoint.get('batch_idx', 'N/A')}")
print(f"Training loss at checkpoint: {checkpoint['loss']:.4f}")

In [ ]:
IMAGENET_VAL_SIZE = 50000
batches_per_eval = IMAGENET_VAL_SIZE // 64

DenseNet161.eval()

top1_correct = 0
top5_correct = 0
total = 0

with torch.no_grad():
    for imgs, lbls in tqdm(loader, total=batches_per_eval, desc="Evaluating"):
        imgs, lbls = imgs.to(device), lbls.to(device)
        out = DenseNet161(imgs)

        _, top1_pred = out.topk(1, dim=1)
        _, top5_pred = out.topk(5, dim=1)

        top1_correct += (top1_pred.squeeze(1) == lbls).sum().item()
        top5_correct += (top5_pred == lbls.unsqueeze(1)).any(dim=1).sum().item()
        total += lbls.size(0)

top1_acc = 100 * top1_correct / total
top5_acc = 100 * top5_correct / total

print(f"Images evaluated : {total:,}")
print(f"Top-1 accuracy   : {top1_acc:.2f}%")
print(f"Top-5 accuracy   : {top5_acc:.2f}%")